# Práctico: Aprendizaje Reforzado para Recomendación con EasyRL4Rec

<a href="https://colab.research.google.com/github/PUC-RecSys-Class/RecSysPUC-2026-2/blob/master/practicos/11_Reinforcement_Learning_EasyRL4Rec.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

IIC3633 Sistemas Recomendadores, PUC Chile, 2026-2.

En el práctico anterior trabajamos con **bandits**: el agente elige una acción, recibe una recompensa inmediata y el "mundo" no cambia por lo que hizo. En este práctico damos el paso a **aprendizaje reforzado (RL) completo**, donde lo que recomendamos hoy **cambia el estado del usuario** y, con eso, las recompensas futuras.

Usaremos la librería [EasyRL4Rec](https://github.com/chongminggao/EasyRL4Rec) ([paper](https://arxiv.org/abs/2402.15164)) para entrenar y comparar tres familias de métodos:

| Familia | Método que usaremos | Idea central |
|---|---|---|
| **Value-based** | DQN | Aprender $Q(s,a)$ y recomendar el ítem con mayor valor |
| **Policy-based** | A2C (actor-critic) | Aprender directamente una distribución $\pi(a \mid s)$ sobre ítems |
| **Offline RL** | BCQ | Aprender solo desde logs históricos, sin interactuar con usuarios |

# Introducción EasyRL4Rec

EasyRL4Rec es una biblioteca de código abierto presentada en SIGIR 2024 por Gao y colaboradores (USTC, Tsinghua y Zhejiang) para bajar la barrera de entrada a los recomendadores con aprendizaje reforzado. Nace de un diagnóstico: cada trabajo del área construía su propio simulador y sus propias métricas, lo que hacía casi imposible reproducir resultados. Ofrece entornos livianos sobre cinco datasets públicos, más de quince algoritmos (de DQN y PPO a CQL, SQN y DORL) y un estándar de evaluación centrado en resultados acumulados.

![Arquitectura EasyRL4Rec](https://raw.githubusercontent.com/PUC-RecSys-Class/RecSysPUC-2026-2/master/practicos/easyRL4Rec.jpg)

Su arquitectura tiene cuatro módulos: Environment convierte un dataset estático en un entorno interactivo, Policy elige la acción, StateTracker codifica el historial como estado y Collector orquesta la interacción.

## 1. Setup

In [ ]:
import os

EASYRL_COMMIT = "95d0acfa55e2150d0775ebd41de0826272edcc94"
TIANSHOU_COMMIT = "d5337368998a93ff2c65f4bb4e36f19e5d624707"

if os.path.basename(os.getcwd()) != "EasyRL4Rec":
    if not os.path.exists("EasyRL4Rec"):
        !git clone -q https://github.com/chongminggao/EasyRL4Rec.git
        !git -C EasyRL4Rec checkout -q {EASYRL_COMMIT}
        !git clone -q https://github.com/yuyq18/tianshou.git EasyRL4Rec/src/tianshou
        !git -C EasyRL4Rec/src/tianshou checkout -q {TIANSHOU_COMMIT}
    os.chdir("EasyRL4Rec")

%pip install -q gymnasium pettingzoo logzero
print("Directorio de trabajo:", os.getcwd())

In [ ]:
# Descarga del dataset Coat (Schnabel et al., 2016) + matriz "pseudo ground-truth" de RL4Rec (Huang et al., 2020)
import io, shutil, urllib.request, zipfile

COAT_DIR = "data/Coat/data_raw"
GT_PATH = f"{COAT_DIR}/RL4Rec_data/coat_pseudoGT_ratingM.ascii"

if not os.path.exists(GT_PATH):
    raw = urllib.request.urlopen("https://www.cs.cornell.edu/~schnabts/mnar/coat.zip").read()
    with zipfile.ZipFile(io.BytesIO(raw)) as z:
        for name in ["coat/train.ascii", "coat/test.ascii",
                     "coat/user_item_features/user_features.ascii",
                     "coat/user_item_features/item_features.ascii"]:
            with z.open(name) as src, open(os.path.join(COAT_DIR, os.path.basename(name)), "wb") as dst:
                shutil.copyfileobj(src, dst)
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/BetsyHJ/RL4Rec/main/data/coat_pseudoGT_ratingM.ascii", GT_PATH)

print(sorted(os.listdir(COAT_DIR)))

In [ ]:
# Parche de compatibilidad con pandas 3 (no es necesario leer esta celda)
import pathlib


def patch_easyrl4rec(root="."):
    root = pathlib.Path(root)
    # Compatibilidad con pandas >= 3 (copy-on-write y dtypes estrictos)
    for p in (root / "src/core/envs").glob("*/*Data.py"):
        txt = p.read_text().replace("item_pop_df['popularity'].fillna(0, inplace=True)",
                                    "item_pop_df['popularity'] = item_pop_df['popularity'].fillna(0)")
        p.write_text(txt)
    p = root / "examples/policy/policy_offline_utils.py"
    txt = p.read_text().replace(
        "df_train_part.loc[:, args.yfeat] = df_train_part[args.yfeat].apply(Max_Min_Scaler)",
        "df_train_part = df_train_part.copy()\n    df_train_part[args.yfeat] = df_train_part[args.yfeat].astype(float).apply(Max_Min_Scaler)")
    p.write_text(txt)


patch_easyrl4rec()
print("Parche aplicado")

In [ ]:
import sys, random, logging, warnings
sys.path.extend([".", "./src", "./src/DeepCTR-Torch", "./src/tianshou", "./examples/policy"])
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import logzero
logzero.loglevel(logging.WARNING)  # silenciamos los logs internos de la librería
logging.getLogger().setLevel(logging.ERROR)

from gymnasium.spaces import Discrete

from src.core.envs.Coat.CoatEnv import CoatEnv
from src.core.collector.collector import Collector
from src.core.collector.collector_set import CollectorSet
from src.core.policy.RecPolicy import RecPolicy
from src.tianshou.tianshou.data import VectorReplayBuffer
from src.tianshou.tianshou.utils.net.common import Net, ActorCritic
from src.tianshou.tianshou.utils.net.discrete import Actor, Critic
from src.tianshou.tianshou.policy import DQNPolicy, A2CPolicy, DiscreteBCQPolicy

from src.core.util.data import get_env_args
from policy_utils import get_args_all, prepare_user_model, prepare_train_test_envs, prepare_test_envs, setup_state_tracker
from policy_offline_utils import get_args_offline, prepare_buffer_via_offline_data


def set_seeds(seed=2023):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "| device:", DEVICE)

## 2. De bandits a RL: recomendación como MDP

Un **Markov Decision Process (MDP)** se define por $(\mathcal{S}, \mathcal{A}, P, R, \gamma)$. En recomendación:

| Elemento | En recomendación |
|---|---|
| Estado $s_t$ | Lo que sabemos del usuario en el turno $t$: sus últimas interacciones y recompensas |
| Acción $a_t$ | El ítem que recomendamos (aquí 1 de 300 chaquetas) |
| Recompensa $r_t$ | Feedback del usuario (aquí, el rating de 1 a 5) |
| Transición $P(s_{t+1} \mid s_t, a_t)$ | Cómo cambia el usuario después de ver la recomendación (por ejemplo, se aburre) |
| Factor de descuento $\gamma$ | Cuánto nos importa el futuro: $\gamma=0$ es miope (un bandit), $\gamma \to 1$ es largo plazo |

El objetivo es maximizar la suma descontada de las recompensas, conocida como **retorno**:

$$G_t = \sum_{k=0}^{T-t} \gamma^k \, r_{t+k}$$


### El dataset Coat

[Coat](https://www.cs.cornell.edu/~schnabts/mnar/) (Schnabel et al., 2016) es un dataset pequeño de **chaquetas** de una tienda online:

- **290 usuarios** y **300 ítems** (chaquetas), con ratings de 1 a 5.
- Cada usuario calificó **24 chaquetas que eligió él mismo** (conjunto de entrenamiento, con sesgo de selección) y **16 chaquetas elegidas al azar** (conjunto de test, sin sesgo).
- Tiene features de usuarios (género, edad, ubicación, interés por la moda) y de ítems (género, tipo de chaqueta, color, si aparecía en la portada).

Para usarlo como entorno de RL necesitamos saber qué rating daría **cualquier** usuario a **cualquier** ítem, pero solo observamos 40 de los 300 ratings de cada usuario. Por eso EasyRL4Rec usa la matriz *pseudo ground-truth* de [RL4Rec](https://github.com/BetsyHJ/RL4Rec) (Huang et al., 2020), donde los ratings faltantes se completaron con un modelo entrenado sobre los ratings observados. Esa matriz completa hace el papel del **usuario "real"** durante la evaluación.

### El entorno `CoatEnv`

EasyRL4Rec convierte un dataset en un entorno [Gymnasium](https://gymnasium.farama.org/):

- `mat`: matriz usuario × ítem **completa** de ratings. Para Coat es una matriz *pseudo ground-truth* (los ratings faltantes se completaron con un modelo), así que podemos "preguntarle" al usuario simulado por cualquier ítem.
- `reset()`: elige un usuario al azar e inicia una sesión.
- `step(item)`: entrega `rating = mat[user, item]` como recompensa y decide si el usuario **abandona**.
- **Mecanismo de abandono:** el usuario se va si el ítem recomendado es **demasiado parecido** (distancia < `leave_threshold`) a alguno de los últimos `num_leave_compute` ítems recomendados. La sesión también termina al llegar a `max_turn` turnos.

In [ ]:
mat, df_item, mat_distance = CoatEnv.load_env_data()
print("Matriz de ratings (usuarios x ítems):", mat.shape)
print("Distribución de ratings:", dict(zip(*np.unique(mat, return_counts=True))))

# Parámetros por defecto que usa EasyRL4Rec para Coat
ENV_KWARGS = dict(mat=mat, df_item=df_item, mat_distance=mat_distance,
                  num_leave_compute=7, leave_threshold=6, max_turn=30, random_init=False)
env = CoatEnv(**ENV_KWARGS)

set_seeds(0)
state, info = env.reset()
print("\nEstado inicial (usuario, último ítem):", state)   # el ítem 300 representa "ninguno todavía"
for item in [10, 42, 42]:
    state, reward, terminated, truncated, info = env.step(item)
    print(f"Recomendamos {item:3d} -> rating={reward}, terminó={terminated}, retorno acumulado={info['cum_reward']}")

Recomendar el **mismo ítem dos veces** hace que el usuario abandone (la distancia de un ítem consigo mismo es 0).

Comparemos algunas políticas simples. `greedy_policy` es un **oráculo**: conoce el rating verdadero y siempre elige el mejor ítem.

In [ ]:
def simulate(policy, n_sessions=300, seed=0):
    # Corre n_sessions en CoatEnv con una política (user, history) -> ítem
    # y retorna el retorno medio, el largo medio de sesión y el rating medio por turno
    set_seeds(seed)
    env = CoatEnv(**ENV_KWARGS)
    returns, lengths = [], []
    for _ in range(n_sessions):
        (user, _), _ = env.reset()
        history, done = [], False
        while not done:
            item = policy(user, history)
            _, r, terminated, truncated, _ = env.step(item)
            history.append(item)
            done = terminated or truncated
        returns.append(env.cum_reward)
        lengths.append(len(history))
    R, L = np.mean(returns), np.mean(lengths)
    return {"R_tra": R, "len_tra": L, "ctr": R / L}


def random_policy(user, history):
    return np.random.randint(mat.shape[1])

def greedy_policy(user, history):
    # Siempre el ítem con mayor rating (miope, puede repetir)
    return int(np.argmax(mat[user]))

def greedy_no_repeat_policy(user, history):
    # El mejor ítem que aún no se ha recomendado
    scores = mat[user].astype(float).copy()
    scores[history] = -np.inf
    return int(np.argmax(scores))

baselines = pd.DataFrame({
    "Aleatoria": simulate(random_policy),
    "Greedy (oráculo)": simulate(greedy_policy),
    "Greedy sin repetir": simulate(greedy_no_repeat_policy),
}).T
baselines.round(2)

Las métricas que usaremos durante todo el práctico son las mismas que usa EasyRL4Rec:
- `R_tra`: retorno acumulado por sesión (trayectoria). **Es la métrica principal.**
- `len_tra`: largo medio de la sesión (cuántos turnos aguanta el usuario).
- `ctr`: recompensa media por turno, es decir, `R_tra / len_tra`.

## 3. Componentes de EasyRL4Rec

Antes de entrenar, veamos cómo se arma un experimento. El pipeline (con **user model**) es:

![pipeline](https://raw.githubusercontent.com/chongminggao/EasyRL4Rec/main/assets/pipeline1.png)

| Componente | Rol | En el código |
|---|---|---|
| **User Model** | Modelo supervisado (DeepFM) entrenado con los logs. Predice la recompensa de cualquier par (usuario, ítem) y actúa como **simulador** para entrenar. Es análogo al *reward model* de RLHF. | `EnsembleModel` |
| **Entornos** | *Entrenamiento*: `BaseSimulatedEnv`, que usa las recompensas **predichas** por el user model. *Evaluación*: `CoatEnv`, que usa la matriz "verdadera". Nunca entrenamos con el entorno de test. | `train_envs`, `test_envs_dict` |
| **StateTracker** | Red que convierte la historia reciente `[(ítem, recompensa), ...]` en un vector de estado $s_t$. Opciones: `avg`, `gru`, `caser`, `sasrec`, `nextitnet`. Se entrena junto con la política. | `StateTracker*` |
| **Policy** | El algoritmo de RL (DQN, A2C, BCQ...) de Tianshou, envuelto en `RecPolicy`, que agrega el manejo de ítems ya recomendados. | `RecPolicy` |
| **Collector** | Hace interactuar la política con varios entornos en paralelo y guarda cada transición en el Buffer. | `Collector`, `CollectorSet` |
| **Buffer** | Memoria de transiciones $(o_t, a_t, r_t, o_{t+1}, \text{terminated})$, organizada por trayectorias. | `VectorReplayBuffer` |

![buffer](https://raw.githubusercontent.com/chongminggao/EasyRL4Rec/main/assets/buffer-2.jpg)

En la figura, cada fila es el sub-buffer de **un entorno** (los entornos corren en paralelo) y cada cuadrado es **una transición**. Los **colores identifican trayectorias**: los cuadrados consecutivos del mismo color pertenecen a la misma sesión de un usuario, y un cambio de color (marcado con *start*) indica que la sesión anterior terminó y el entorno empezó una nueva. Los colores no representan usuarios ni recompensas, solo sirven para distinguir dónde empieza y termina cada trayectoria.

**Evaluación.** EasyRL4Rec evalúa cada política en tres modos:
- **FB** (*Free*): se permiten repeticiones y el usuario puede abandonar.
- **NX_0**: se prohíbe repetir ítems y el usuario puede abandonar.
- **NX_10**: se prohíbe repetir ítems y las sesiones duran exactamente 10 turnos, sin abandono. Mide calidad de los ítems sin el efecto del aburrimiento.

### 3.1 Entrenar el User Model (simulador)

Usamos el script de la librería. Entrenamos **un** DeepFM durante 3 épocas

In [ ]:
UM_NAME = "um"
if not os.path.exists(f"saved_models/CoatEnv-v0/DeepFM/params/[{UM_NAME}]_params.pickle"):
    !python examples/usermodel/run_DeepFM_ensemble.py --env CoatEnv-v0 --seed 2023 --epoch 3 --n_models 1 --loss "pointneg" --message {UM_NAME} 2>&1 | grep -E "RMSE|saved" | tail -n 2
print("User model listo")

### 3.2 Armar los argumentos y los entornos

Los scripts de EasyRL4Rec reciben todos sus hiperparámetros por línea de comandos (`argparse`). La función `build_args` reutiliza esos mismos parsers desde el notebook: construye los argumentos por defecto de la librería para Coat y luego sobrescribe los valores que le pasemos.

In [ ]:
def build_args(paradigm, **overrides):
    # paradigm: "offpolicy" (DQN), "onpolicy" (A2C) u "offline" (BCQ).
    # Usa los mismos defaults que los scripts de la librería y luego aplica overrides.
    sys.argv = ["notebook", "--env", "CoatEnv-v0", "--read_message", UM_NAME]
    args = get_args_all(paradigm)
    if paradigm == "offline":
        args = get_args_offline(args)
    args = get_env_args(args)
    defaults = dict(
        seed=2023, device=DEVICE,
        which_tracker="avg", reward_handle="cat", window_size=3,
        training_num=10,     # entornos de entrenamiento en paralelo
        test_num=50,         # entornos (sesiones) de evaluación
        force_length=10,     # largo de sesión del modo NX_10
        gamma=0.9,
    )
    defaults.update(overrides)
    for k, v in defaults.items():
        setattr(args, k, v)
    return args


set_seeds()
args_base = build_args("offpolicy")
ensemble_models = prepare_user_model(args_base)   # carga el DeepFM entrenado
env, dataset, train_envs, test_envs_dict = prepare_train_test_envs(args_base, ensemble_models)

print("Entornos de entrenamiento (simulados con el user model):", len(train_envs))
print("Modos de evaluación:", {k: len(v) for k, v in test_envs_dict.items()})

# ¿Qué tan bueno es el simulador? Comparamos la matriz que predice el user model con la "verdadera"
pred_mat = train_envs.get_env_attr("predicted_mat")[0]
print(f"\nCorrelación entre las recompensas predichas y las verdaderas: {np.corrcoef(pred_mat.ravel(), mat.ravel())[0, 1]:.3f}")

La correlación no es perfecta: **el agente se entrena en un simulador imperfecto** y se evalúa en el entorno "real". Esta brecha (*sim-to-real gap*) es uno de los problemas centrales de RL para recomendación.

### 3.3 Funciones comunes: StateTracker, evaluación y gráficos

- `create_state_tracker(args)`: crea un StateTracker nuevo para cada método, inicializado con los embeddings de ítems del user model. También fija `args.state_dim` (dimensión de $s_t$) y `args.action_shape` (número de ítems).
- `evaluate(policy, test_collector_set)`: corre `test_num` sesiones en cada modo (FB, NX_0, NX_10) en el entorno **real** y retorna `R_tra`, `len_tra` y `ctr`.

In [ ]:
def create_state_tracker(args, buffer=None):
    # Para offline RL las normalizaciones de recompensa se calculan sobre el buffer de logs
    if buffer is not None:
        return setup_state_tracker(args, ensemble_models, env, buffer, test_envs_dict, use_buffer_in_train=True)
    return setup_state_tracker(args, ensemble_models, env, train_envs, test_envs_dict)


def evaluate(policy, test_collector_set, n_sessions):
    # Ejecuta n_sessions por modo con la política en modo evaluación
    test_collector_set.reset_env()
    test_collector_set.reset_buffer()
    policy.eval()
    res = test_collector_set.collect(n_episode=n_sessions)
    metrics = {}
    for mode, prefix in [("FB", ""), ("NX_0", "NX_0_"), ("NX_10", "NX_10_")]:
        R, L = res[prefix + "rew"], res[prefix + "len"]
        metrics[mode] = {"R_tra": R, "len_tra": L, "ctr": R / L}
    return metrics


def print_epoch(epoch, loss, m):
    print(f"Época {epoch:2d} | pérdida {loss:8.4f} | "
          f"FB: R_tra={m['FB']['R_tra']:5.2f} len={m['FB']['len_tra']:5.2f} | "
          f"NX_0: R_tra={m['NX_0']['R_tra']:5.2f} len={m['NX_0']['len_tra']:5.2f} | "
          f"NX_10: ctr={m['NX_10']['ctr']:.3f}")


def to_dataframe(history):
    # history: lista de (época, métricas) -> DataFrame en formato largo
    rows = [{"epoch": e, "mode": mode, **vals} for e, m in history for mode, vals in m.items()]
    return pd.DataFrame(rows)


def plot_histories(histories):
    # histories: dict nombre del método -> history
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    # Segundo eje Y en el panel FB para ver el largo de las sesiones (línea punteada)
    ax_len = axes[0].twinx()
    axes[0].set_zorder(ax_len.get_zorder() + 1)   # la leyenda queda sobre las líneas punteadas
    axes[0].patch.set_visible(False)
    for i, (name, hist) in enumerate(histories.items()):
        color = f"C{i}"
        df = to_dataframe(hist)
        for ax, (mode, metric) in zip(axes, [("FB", "R_tra"), ("NX_0", "R_tra"), ("NX_10", "ctr")]):
            d = df[df["mode"] == mode]
            ax.plot(d["epoch"], d[metric], marker="o", color=color, label=name)
            ax.set_title(f"{mode}: {metric}")
            ax.set_xlabel("época")
        d = df[df["mode"] == "FB"]
        ax_len.plot(d["epoch"], d["len_tra"], ls="--", marker="x", color=color, alpha=0.6)
    axes[0].set_title("FB: R_tra (continua) y len_tra (punteada)")
    axes[0].set_ylabel("R_tra")
    ax_len.set_ylabel("len_tra")
    ax_len.set_ylim(bottom=0)
    # Referencias: greedy (oráculo) permite repetir como FB; greedy sin repetir es comparable a NX_0
    axes[0].axhline(baselines.loc["Greedy (oráculo)", "R_tra"], ls="--", c="gray", label="Greedy (oráculo)")
    axes[0].axhline(baselines.loc["Aleatoria", "R_tra"], ls="-.", c="silver", label="Aleatoria")
    axes[1].axhline(baselines.loc["Greedy sin repetir", "R_tra"], ls=":", c="gray", label="Greedy sin repetir")
    for ax in axes:
        ax.legend()
    plt.tight_layout()
    plt.show()

## 4. Value-based: DQN

**Idea.** Aprender la función de valor acción-estado

$$Q(s,a) = \mathbb{E}\Big[\sum_k \gamma^k r_{t+k} \;\Big|\; s_t=s,\, a_t=a\Big]$$

y recomendar $a^* = \arg\max_a Q(s,a)$. La red recibe el estado $s_t$ y entrega **un valor $Q$ por cada ítem** (300 salidas).

**Aprendizaje (ecuación de Bellman).** Se minimiza el error entre $Q(s_t,a_t)$ y el *target*:

$$y_t = r_t + \gamma \max_{a'} Q_{\text{target}}(s_{t+1}, a')$$

**Particularidades:**
- **Off-policy con replay buffer:** las transiciones se guardan y se reutilizan muchas veces (minibatches aleatorios). Es eficiente en datos.
- **Exploración ε-greedy:** con probabilidad ε se recomienda un ítem al azar. La política aprendida es **determinista**.
- **Target network:** una copia de la red que se actualiza cada `target_update_freq` pasos, para estabilizar el target.
- Usamos *Double DQN* y *n-step returns* (`n_step=3`), como en los defaults de la librería.
- **Limitación en recomendación:** se necesita una salida por ítem y el $\max$ sobre todo el catálogo **sobreestima** los valores. Con catálogos de millones de ítems esto no escala directamente.

### 4.1 Construcción de la política

In [ ]:
def build_dqn(args, state_tracker):
    # Red Q: estado (state_dim) -> un valor Q por ítem (action_shape)
    net = Net(args.state_dim, args.action_shape, hidden_sizes=args.hidden_sizes, device=args.device).to(args.device)
    # Dos optimizadores: uno para la red Q y otro para el StateTracker (se entrenan juntos)
    optim = [torch.optim.Adam(net.parameters(), lr=args.lr),
             torch.optim.Adam(state_tracker.parameters(), lr=args.lr)]
    policy = DQNPolicy(
        net, optim, state_tracker=state_tracker,
        discount_factor=args.gamma,
        estimation_step=args.n_step,                 # n-step returns
        target_update_freq=args.target_update_freq,  # cada cuánto se copia la target network
        is_double=True,
        action_space=Discrete(args.action_shape),
    )
    policy.set_eps(args.explore_eps)                 # epsilon de la exploración epsilon-greedy
    return RecPolicy(args, policy, state_tracker)


def prepare_collectors(args, rec_policy, with_train=True):
    # Collector de entrenamiento: interactúa con los entornos SIMULADOS y llena el buffer.
    # CollectorSet de evaluación: un collector por modo (FB, NX_0, NX_10) sobre los entornos REALES.
    train_collector = None
    if with_train:
        buffer = VectorReplayBuffer(args.buffer_size, buffer_num=len(train_envs))
        train_collector = Collector(rec_policy, train_envs, buffer,
                                    exploration_noise=args.exploration_noise,
                                    remove_recommended_ids=args.remove_recommended_ids)
    test_collector_set = CollectorSet(rec_policy, test_envs_dict, args.buffer_size, args.test_num,
                                      exploration_noise=False, force_length=args.force_length)
    return train_collector, test_collector_set

In [ ]:
set_seeds()
args_dqn = build_args("offpolicy", explore_eps=0.1, target_update_freq=320, lr=1e-3,
                      step_per_collect=100, update_per_step=0.125, batch_size=64)
st_dqn = create_state_tracker(args_dqn)
print("Dimensión del estado:", args_dqn.state_dim, "| número de ítems (acciones):", args_dqn.action_shape)
dqn = build_dqn(args_dqn, st_dqn)
train_collector_dqn, test_collectors_dqn = prepare_collectors(args_dqn, dqn)
dqn.policy.model

### 4.2 Mirando dentro del Collector y del Buffer

Recolectamos 100 pasos de interacción (10 por entorno) con la política aún **sin entrenar** y miramos lo que quedó guardado. Cada fila es una transición: `obs = (user, ítem anterior)`, `act` = ítem recomendado, `rew` = recompensa **del simulador** y `terminated` indica si el usuario abandonó.

Fíjate en qué pasa cuando la política recomienda dos veces el mismo ítem.

In [ ]:
def show_buffer(buffer, n=10):
    # Muestra las primeras n transiciones del primer sub-buffer (el de un entorno)
    sub = buffer.buffers[0]
    k = min(n, len(sub))
    # Ojo: al inicio de cada episodio el Collector guarda en obs el (usuario, ítem) del episodio ANTERIOR.
    # Por eso tomamos el usuario de obs_next (que sí es correcto) y usamos is_start para marcar
    # que al inicio del episodio todavía no hay ítem anterior.
    item_prev = np.where(sub.is_start[:k], "ninguno", sub.obs[:k, 1].astype(str))
    return pd.DataFrame({
        "usuario": sub.obs_next[:k, 0], "item_anterior": item_prev, "act (ítem recomendado)": sub.act[:k],
        "rew": np.round(sub.rew[:k], 3), "terminated": sub.terminated[:k],
    })

res = train_collector_dqn.collect(n_step=100)
print({k: res[k] for k in ["n/st", "n/ep"]})
show_buffer(train_collector_dqn.buffer)

### 4.3 Loop de entrenamiento off-policy

Cada época repite dos pasos hasta completar `steps_per_epoch` interacciones:

1. **Interactuar:** `train_collector.collect(n_step=...)` ejecuta la política (con ε-greedy) en los entornos simulados y agrega las transiciones al buffer.
2. **Aprender:** se muestrean minibatches **aleatorios de todo el buffer**, incluyendo datos viejos generados por versiones anteriores de la política (eso es *off-policy*). Con `policy.update(...)` se calcula el target de Bellman y se actualizan la red Q y el StateTracker.

Al final de cada época evaluamos en el entorno real.

In [ ]:
def train_offpolicy(policy, train_collector, test_collector_set, args, n_epochs, steps_per_epoch):
    history = [(0, evaluate(policy, test_collector_set, args.test_num))]
    for epoch in range(1, n_epochs + 1):
        policy.train()
        steps, losses = 0, []
        while steps < steps_per_epoch:
            # 1) Interactuar con el simulador y guardar en el buffer
            res = train_collector.collect(n_step=args.step_per_collect)
            steps += res["n/st"]
            # 2) Varias actualizaciones con minibatches muestreados del buffer
            for _ in range(round(args.update_per_step * res["n/st"])):
                info = policy.update(args.batch_size, train_collector.buffer)
                losses.append(info["loss"])
        metrics = evaluate(policy, test_collector_set, args.test_num)
        history.append((epoch, metrics))
        print_epoch(epoch, np.mean(losses), metrics)
    return history


hist_dqn = train_offpolicy(dqn, train_collector_dqn, test_collectors_dqn, args_dqn, n_epochs=10, steps_per_epoch=2000)

In [ ]:
plot_histories({"DQN": hist_dqn})

**Observa:**
- En el modo **FB** (con repeticiones permitidas), DQN suele tener sesiones muy cortas: como su política es determinista ($\arg\max$), tiende a recomendar una y otra vez los mismos ítems "de alto valor Q", y el usuario abandona.
- En **NX_0** (sin repeticiones) le va mucho mejor, aunque con mucha variación entre épocas. Una explicación es la **sobreestimación** típica del $\max$ en Q-learning: la red cree que repetir el ítem de mayor Q es buena idea porque no aprendió bien el costo del abandono.

## 5. Policy-based: A2C (Advantage Actor-Critic)

**Idea.** En lugar de aprender valores y derivar la política, aprendemos **directamente** la política $\pi_\theta(a \mid s)$: una distribución softmax sobre los 300 ítems. Para recomendar, **muestreamos** un ítem de esa distribución.

**Policy gradient (REINFORCE):**

$$\nabla_\theta J(\theta) = \mathbb{E}\big[\nabla_\theta \log \pi_\theta(a_t \mid s_t)\; A_t\big]$$

**Actor-critic:** un **actor** $\pi_\theta(a|s)$ y un **crítico** $V_\phi(s)$ que estima cuánto retorno esperamos desde $s$. La **ventaja** $A_t = G_t - V_\phi(s_t)$ dice si la acción fue mejor o peor de lo esperado, y reduce la varianza del gradiente respecto a REINFORCE puro.

**Particularidades:**
- **On-policy:** el gradiente solo es válido con datos de la política actual. Después de cada actualización **el buffer se vacía**, así que es menos eficiente en datos que DQN.
- **Política estocástica:** la exploración viene "gratis" al muestrear. Además, un bonus de **entropía** (`ent_coef`) evita que la distribución colapse en un solo ítem.
- Recomendar muestreando produce naturalmente **más diversidad**, algo muy valioso cuando el usuario se aburre con ítems parecidos.
- Es la familia detrás del caso industrial más citado: REINFORCE con corrección off-policy en YouTube ([Chen et al., 2019](https://arxiv.org/abs/1812.02353)). PPO es una variante más estable de esta misma familia.

In [ ]:
def build_a2c(args, state_tracker):
    net = Net(args.state_dim, hidden_sizes=args.hidden_sizes, device=args.device)  # cuerpo compartido
    actor = Actor(net, args.action_shape, device=args.device).to(args.device)      # pi(a|s): logits por ítem
    critic = Critic(net, device=args.device).to(args.device)                       # V(s): un escalar
    optim = [torch.optim.Adam(ActorCritic(actor, critic).parameters(), lr=args.lr),
             torch.optim.Adam(state_tracker.parameters(), lr=args.lr)]
    policy = A2CPolicy(
        actor, critic, optim, torch.distributions.Categorical,  # distribución sobre ítems
        state_tracker=state_tracker,
        discount_factor=args.gamma,
        gae_lambda=1.0,            # lambda=1: la ventaja usa retornos Monte Carlo
        vf_coef=0.5,               # peso de la pérdida del crítico
        ent_coef=args.ent_coef,    # bonus de entropía (exploración)
        action_space=Discrete(args.action_shape),
        action_bound_method="", action_scaling=False,
    )
    return RecPolicy(args, policy, state_tracker)

In [ ]:
set_seeds()
args_a2c = build_args("onpolicy", lr=0.005, ent_coef=0.01, episode_per_collect=50, batch_size=1024, repeat_per_collect=1)
st_a2c = create_state_tracker(args_a2c)
a2c = build_a2c(args_a2c, st_a2c)
train_collector_a2c, test_collectors_a2c = prepare_collectors(args_a2c, a2c)

### Loop de entrenamiento on-policy

Comparado con DQN cambian dos cosas:

1. Se recolectan **episodios completos** (`n_episode`), porque necesitamos los retornos $G_t$ para calcular la ventaja.
2. Después de actualizar, `reset_buffer()` **descarta** todos los datos: la próxima actualización usará solo datos de la política nueva.

In [ ]:
def train_onpolicy(policy, train_collector, test_collector_set, args, n_epochs, steps_per_epoch):
    history = [(0, evaluate(policy, test_collector_set, args.test_num))]
    for epoch in range(1, n_epochs + 1):
        policy.train()
        steps, losses = 0, []
        while steps < steps_per_epoch:
            # 1) Recolectar episodios completos con la política ACTUAL
            res = train_collector.collect(n_episode=args.episode_per_collect)
            steps += res["n/st"]
            # 2) Una actualización de actor y crítico con todo lo recolectado
            info = policy.update(0, train_collector.buffer, batch_size=args.batch_size, repeat=args.repeat_per_collect)
            losses.extend(info["loss"])
            # 3) Descartar los datos: ya no corresponden a la política actual
            train_collector.reset_buffer(keep_statistics=True)
        metrics = evaluate(policy, test_collector_set, args.test_num)
        history.append((epoch, metrics))
        print_epoch(epoch, np.mean(losses), metrics)
    return history


hist_a2c = train_onpolicy(a2c, train_collector_a2c, test_collectors_a2c, args_a2c, n_epochs=10, steps_per_epoch=2000)

In [ ]:
plot_histories({"DQN": hist_dqn, "A2C": hist_a2c})

**Observa:** A2C logra sesiones mucho más largas en el modo FB. Al ser estocástica, la política no repite el mismo ítem y el usuario no se aburre tan rápido. El `ctr` (rating por turno) puede ser menor que el de greedy, pero el **retorno total** es mucho mayor.

## 6. Offline RL: BCQ (Batch-Constrained Q-learning)

**Motivación.** En la práctica, desplegar una política que explora al azar con usuarios reales es caro o riesgoso, y los simuladores son imperfectos (vimos la correlación de la sección 3.2). **Offline RL** aprende **solo desde logs históricos**, sin interactuar con el entorno durante el entrenamiento.

![offline](https://raw.githubusercontent.com/chongminggao/EasyRL4Rec/main/assets/pipeline2.png)

**El problema: *distribution shift* / error de extrapolación.** Si aplicamos DQN directamente sobre logs, el $\max_{a'} Q(s', a')$ elige ítems que **nunca aparecieron** en los datos. Sus valores $Q$ son pura extrapolación (normalmente sobreestimada) y nadie los corrige, porque no podemos probarlos.

**BCQ (versión discreta).** Además de la red $Q$, entrena una red de **imitación** $G_\omega(a \mid s)$ que estima qué ítems habría recomendado la política que generó los logs (*behavior policy*). Al elegir acciones, solo considera ítems "plausibles":

$$a^* = \arg\max_{a \,:\, G_\omega(a|s) / \max_{\hat a} G_\omega(\hat a|s) \;>\; \tau} Q(s,a)$$

- $\tau = 0$: es DQN offline (sin restricción).
- $\tau \to 1$: es *behavior cloning* (imita los logs).

**Particularidades:**
- No hay `train_collector`: el buffer se construye **una sola vez** a partir del dataset de entrenamiento (ratings de Coat), agrupando las interacciones de cada usuario en una trayectoria.
- La calidad queda limitada por la cobertura de los logs.
- Otros métodos de esta familia: CQL (penaliza los Q de acciones fuera de los datos) y CRR (imitación ponderada por la ventaja). Ambos están en EasyRL4Rec.

### 6.1 Construir el buffer desde los logs

In [ ]:
set_seeds()
args_bcq = build_args("offline", lr=1e-3, n_step=3, target_update_freq=320,
                      unlikely_action_threshold=0.6, imitation_logits_penalty=0.01,
                      explore_eps=0.0, batch_size=64, construction_method="normal")

# Convierte el dataset de entrenamiento de Coat (logs de ratings) en trayectorias dentro de un buffer
_, _, _, log_buffer = prepare_buffer_via_offline_data(args_bcq)
print("Transiciones en el buffer offline:", len(log_buffer))
show_buffer(log_buffer)

Las recompensas ahora vienen de los **ratings reales** de los logs (normalizados a [0, 1]), no del user model.

### 6.2 Construcción de BCQ y loop de entrenamiento offline

El loop es el más simple de los tres: **solo** se muestrean minibatches del buffer fijo y se actualiza. No hay ningún paso de interacción con el entorno.

In [ ]:
def build_bcq(args, state_tracker, buffer):
    net = Net(args.state_dim, args.hidden_sizes[0], device=args.device)
    q_net = Actor(net, args.action_shape, hidden_sizes=args.hidden_sizes, device=args.device).to(args.device)          # Q(s,a)
    imitation_net = Actor(net, args.action_shape, hidden_sizes=args.hidden_sizes, device=args.device).to(args.device)  # G(a|s)
    optim = [torch.optim.Adam(ActorCritic(q_net, imitation_net).parameters(), lr=args.lr),
             torch.optim.Adam(state_tracker.parameters(), lr=args.lr)]
    policy = DiscreteBCQPolicy(
        q_net, imitation_net, optim,
        args.gamma, args.n_step, args.target_update_freq,
        args.explore_eps,
        args.unlikely_action_threshold,   # tau: umbral de plausibilidad
        args.imitation_logits_penalty,
        state_tracker=state_tracker, buffer=buffer,
        action_space=Discrete(args.action_shape),
    )
    policy.set_eps(args.explore_eps)
    return RecPolicy(args, policy, state_tracker)


def train_offline(policy, buffer, test_collector_set, args, n_epochs, updates_per_epoch):
    history = [(0, evaluate(policy, test_collector_set, args.test_num))]
    for epoch in range(1, n_epochs + 1):
        policy.train()
        losses = []
        for _ in range(updates_per_epoch):
            # Solo aprender desde el buffer fijo de logs: sin interacción
            info = policy.update(args.batch_size, buffer)
            losses.append(info["loss"])
        metrics = evaluate(policy, test_collector_set, args.test_num)
        history.append((epoch, metrics))
        print_epoch(epoch, np.mean(losses), metrics)
    return history

In [ ]:
st_bcq = create_state_tracker(args_bcq, buffer=log_buffer)
bcq = build_bcq(args_bcq, st_bcq, log_buffer)
_, test_collectors_bcq = prepare_collectors(args_bcq, bcq, with_train=False)

hist_bcq = train_offline(bcq, log_buffer, test_collectors_bcq, args_bcq, n_epochs=10, updates_per_epoch=250)

**Observa:** BCQ suele tener el **mejor `ctr` en NX_10**: aprendió de los logs qué ítems les gustan a los usuarios. Pero sus sesiones son cortas. Como nunca interactúa con el entorno, no puede descubrir que recomendar ítems parecidos aburre al usuario: solo sabe lo que aparece en los datos.

## 7. Comparación

In [ ]:
plot_histories({"DQN": hist_dqn, "A2C": hist_a2c, "BCQ": hist_bcq})

def summarize(histories):
    # Métricas de la última época de cada método (las políticas de referencia están en la sección 2)
    rows = {}
    for name, hist in histories.items():
        m = hist[-1][1]
        rows[name] = {(mode, k): m[mode][k] for mode in m for k in ["R_tra", "len_tra", "ctr"]}
    return pd.DataFrame(rows).T.round(2)

summarize({"DQN": hist_dqn, "A2C": hist_a2c, "BCQ": hist_bcq})

# Resumen
| | DQN (value-based) | A2C (policy-based) | BCQ (offline) |
|---|---|---|---|
| Qué aprende | $Q(s,a)$ | $\pi(a \mid s)$ y $V(s)$ | $Q(s,a)$ + imitación $G(a \mid s)$ |
| Política | Determinista ($\arg\max$) + ε-greedy | Estocástica (muestreo) | Determinista restringida a acciones plausibles |
| Datos | Off-policy: replay buffer reutilizado | On-policy: el buffer se descarta | Buffer fijo de logs, sin interacción |
| Eficiencia en datos | Alta | Baja | No necesita interactuar |
| Riesgo típico | Sobreestimación de Q | Alta varianza, necesita más muestras | Limitado por la cobertura de los logs |


## 8. Actividades

Las funciones `build_args`, `create_state_tracker`, `build_dqn`/`build_a2c`/`build_bcq`, `prepare_collectors` y `train_*` ya están definidas y se pueden reutilizar directamente.

### Actividad 1: El rol de $\gamma$ (value-based)

Entrena DQN con `gamma=0.0` y compáralo con el DQN de la sección 4 (`gamma=0.9`).

1. ¿A qué tipo de algoritmo, visto en la clase anterior, equivale DQN con $\gamma = 0$? ¿Qué está aprendiendo $Q(s,a)$ en ese caso?
2. ¿Cómo cambian `R_tra`, `len_tra` y `ctr` en los modos FB y NX_0? ¿Coincide con lo que esperabas en teoría?
3. Si el resultado no coincide con la teoría, propón una explicación. *Pista: con $\gamma > 0$ el target $r + \gamma \max_{a'} Q(s',a')$ depende de la propia red, y entrenamos solo 10 épocas en un simulador imperfecto.*

In [ ]:
set_seeds()
args_g0 = build_args("offpolicy", explore_eps=0.1, target_update_freq=320, lr=1e-3,
                     step_per_collect=100, update_per_step=0.125, batch_size=64,
                     gamma=0.0)
# TODO: crea el state tracker, la política DQN y los collectors (ver sección 4),
#       entrena 10 épocas con train_offpolicy y grafica junto a hist_dqn con plot_histories({...})

### Actividad 2: ¿Por qué importa que la política sea estocástica?

A2C recomienda **muestreando** de $\pi(a|s)$. Tianshou permite evaluar la misma política de forma **determinista**, recomendando siempre $\arg\max_a \pi(a|s)$. Evalúa el modelo `a2c` ya entrenado de ambas formas (no hay que reentrenar).

1. Compara `R_tra`, `len_tra` y `ctr` en los modos FB y NX_0. ¿Qué política de la sección 2 se parece a A2C determinista?
2. DQN es determinista por diseño. ¿Qué nos dice este experimento sobre los resultados de DQN en el modo FB?

In [ ]:
set_seeds()
stochastic = evaluate(a2c, test_collectors_a2c, args_a2c.test_num)

a2c.policy._deterministic_eval = True    # en evaluación usa argmax_a pi(a|s) en vez de muestrear
# TODO: evalúa de nuevo con evaluate(...) y guarda el resultado en `deterministic`
a2c.policy._deterministic_eval = False   # dejamos la política como estaba

# TODO: compara ambos resultados, por ejemplo con pd.DataFrame({...}) por modo

### Actividad 3: ¿Cuánto restringir en offline RL?

Entrena BCQ con `unlikely_action_threshold` $\tau \in \{0.0, 0.3, 0.9\}$ (más el 0.6 ya entrenado).

1. ¿Qué método obtienes en los extremos $\tau = 0$ y $\tau \approx 1$?
2. ¿Qué valor funciona mejor? Explica el resultado en términos de *distribution shift*.

*Tip: `log_buffer` ya está construido y puedes reutilizarlo.*

In [ ]:
hist_tau = {"BCQ tau=0.6": hist_bcq}
for tau in [0.0, 0.3, 0.9]:
    set_seeds()
    args_t = build_args("offline", lr=1e-3, n_step=3, target_update_freq=320,
                        unlikely_action_threshold=tau, imitation_logits_penalty=0.01,
                        explore_eps=0.0, batch_size=64)
    # TODO: crea el state tracker (con buffer=log_buffer), la política BCQ y el CollectorSet de test (ver sección 6),
    #       entrena 10 épocas con train_offline y guarda el historial en hist_tau[f"BCQ tau={tau}"]

# plot_histories(hist_tau)
# summarize(hist_tau)

### Actividad 4: Representación del estado

Entrena A2C cambiando el StateTracker a `which_tracker="gru"` (una GRU sobre la secuencia, como en GRU4Rec) y compáralo con el A2C de la sección 5, que usa `avg`.

1. ¿Mejora el retorno? ¿Cuánto más lento es entrenar?
2. ¿Qué información sobre el usuario captura una GRU que el promedio (`avg`) no captura?

In [ ]:
set_seeds()
args_gru = build_args("onpolicy", lr=0.005, ent_coef=0.01, episode_per_collect=50, batch_size=1024,
                      repeat_per_collect=1, which_tracker="gru")
# TODO: crea el state tracker, la política A2C y los collectors (ver sección 5),
#       entrena 10 épocas con train_onpolicy y compara con hist_a2c usando plot_histories({...}) y summarize({...})

### Para seguir explorando

- Otros métodos de la librería: `PPO`, `PG` (REINFORCE), `C51`, `DDPG`/`TD3` (acciones continuas mapeadas al ítem más cercano), `CQL`, `CRR`, `SQN`, y métodos avanzados como `DORL` o recompensas intrínsecas de diversidad y novedad (`examples/advance/`).
- Otros datasets o simuladores: KuaiRec, KuaiRand, MovieLens y Yahoo (requieren descargar el dataset completo de la librería, de 8.6 GB).
- Los scripts de `script/run_example_Coat.sh` muestran los hiperparámetros que usaron los autores.